In [5]:
import sys
print(sys.executable)
import os 
#import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests
from langchain.agents import create_react_agent, AgentExecutor

c:\megha\agentic-ai buppy\langraph-agent\venv\Scripts\python.exe


In [ ]:
from langchain.agents import create_react_agent, AgentExecutor

In [36]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
#os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
# WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

True

In [ ]:
search_tool = TavilySearchResults(max_results=2)
search_tool.invoke("What is the capital of France?")

[{'url': 'https://www.britannica.com/place/France',
  'content': 'The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial centres. A majestic city known as the _ville lumière_, or “city of light,” Paris has often been remade, most famously in the mid-19th century under the command of Georges-Eugène, Baron Haussman, who was committed to Napoleon III’s vision of a modern city free of the choleric swamps and congested alleys of old, with broad avenues and a regular plan. Paris is now a sprawling metropolis, one of Europe’s largest conurbations, but its historic heart can still be traversed in an evening’s walk. Confident that their city stood at the very centre of the world, Parisians were once given to referring to their country as having two parts, Paris and _le désert_, the wasteland [...] Quick Facts\n\nImage 23: France\n\nSee article: flag of France\n\nAudio 1\n\nAudio File: Anthem of France (see article)\n\nHead Of Gov

In [ ]:
@tool
def get_weather(location: str) -> str:
    print("hello", {WEATHERSTACK_API_KEY})
    """
    Get the current weather for a given location using the Weatherstack API.
    """
    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={location}"
    print("hello", {WEATHERSTACK_API_KEY})
    response = requests.get(url)
    data = response.json()
    if "current" in data:
        temperature = data["current"]["temperature"]
        description = data["current"]["weather_descriptions"][0]
        return f"The current weather in {location} is {description} with a temperature of {temperature}°C."
    else:
        return f"Could not retrieve weather data for {location}."

    get_weather("New York")

In [17]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
)

In [20]:
response = llm.invoke("WHwat is the capital of France?")
response.content

'The capital of France is Paris.'

In [22]:
prompt = hub.pull("hwchase17/react")
prompt

c:\megha\agentic-ai buppy\langraph-agent\venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [38]:
tools = [search_tool,get_weather]

In [25]:
agent = create_react_agent(llm, tools, prompt=prompt)

In [28]:
agentexecutor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [39]:
response = agentexecutor.invoke({
    "input": (
        "Find the capital of Gujarat and then find the weather of that city."
    )
})



> Entering new AgentExecutor chain...
To find the capital of Gujarat, I will first identify the city and then search for its current weather.

Action: tavily_search_results_json
Action Input: "capital of Gujarat"
[{'url': 'https://en.wikipedia.org/wiki/Gandhinagar', 'content': 'Wikipedia\n\n# Gandhinagar\n\nGandhinagar (Gujarati: gāndhīnagara, pronounced (//en.wikipedia.org/wiki/Help:IPA/Gujarati "Help:IPA/Gujarati") ⓘ) in India is the capital of the state of Gujarat and of Gandhinagar district. Gandhinagar is located approximately 23 kilometres (14 mi) north of Ahmedabad, on the west central point of the industrial corridor between the megacities of Delhi and Mumbai.'}]The capital of Gujarat is Gandhinagar. Now, I will find the current weather in Gandhinagar.

Action: tavily_search_results_json
Action Input: "current weather in Gandhinagar"
[{'url': 'https://en.climate-data.org/asia/india/gujarat/gandhinagar-5583/t/june-6', 'content': '| 26. June | 31 °C | 88 °F | 35 °C | 96 °F | 27